# West-Med Global Fishing Watch events

Query the [Global Fishing Watch](https://globalfishingwatch.org) Events API (the
bundled `gfw` source) for a slice of the western Mediterranean and save the
matching fishing / port-visit / encounter / loitering / AIS-gap events as an
annotated Parquet file.

GFW is **credential-gated**. Create a token at the
[GFW API portal](https://globalfishingwatch.org/our-apis/tokens) and put it in a
`.env` at the repo root (collekt loads it on import):

```
GFW_API_ACCESS_TOKEN=…
```

> GFW's [Terms of Use](https://globalfishingwatch.org/terms-of-use/) restrict the
> API to **non-commercial** use.

Files are written under `notebooks/cache/` (git-ignored).

## Check the credential is visible

In [ ]:
import os

import collekt
from collekt.core.doctor import run_doctor

for check in run_doctor():
    if check.name.startswith("gfw"):
        print(f"{check.status.value:>5}  {check.name}: {check.message}")

assert os.environ.get("GFW_API_ACCESS_TOKEN"), (
    "No GFW_API_ACCESS_TOKEN found. Add it to a .env at the repo root, then restart the kernel."
)

## Build the request

GFW filters on **overlap**, not on event start: a port visit that merely spans the
window comes back with its true (much wider) `start`/`end`. Keep the region and
window modest — the API is happier with small boxes and short windows, and the
adapter's `limit` truncates rather than paginating.

In [ ]:
from pathlib import Path


def _repo_root(marker: str = "pyproject.toml") -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / marker).exists():
            return candidate
    return here


OUTPUT_ROOT = _repo_root() / "notebooks" / "cache" / "westmed_gfw"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

LIMIT = 1000

config = collekt.DatasetConfig(
    collekt.GFW(
        datasets=[
            "public-global-fishing-events:latest",
            "public-global-port-visits-events:latest",
            "public-global-encounters-events:latest",
            "public-global-loitering-events:latest",
            "public-global-gaps-events:latest",
        ],
        limit=LIMIT,
    )
)
request = collekt.Request(
    region=collekt.Region.from_bbox((2.0, 8.0, 37.0, 41.0)),  # Balearic Sea
    start="2023-06-01",
    end="2023-06-07",
)
request.as_dict(), config.as_dict()

## Dry run

`plan()` shows the exact query the adapter will send without touching the network —
note `end_date` is the day *after* the request's end, because GFW treats it as
exclusive.

In [ ]:
plan = collekt.Fetcher(request=request, config=config, output_dir=OUTPUT_ROOT).plan()
plan.results[0].details["request"]

## Download

In [ ]:
fetcher = collekt.Fetcher(
    request=request,
    config=config,
    output_dir=OUTPUT_ROOT,
    progress=lambda source, message: print(f"[{source}] {message}"),
)
result = fetcher.download()
result

## Inspect the events

The adapter writes one annotated Parquet file for the whole request; read it back
with polars.

In [ ]:
import polars as pl
from IPython.display import display

events = pl.read_parquet(result.files[0]) if result.files else None
if events is None:
    print("No events for this query — widen the region/window, or check the warning above.")
else:
    print(f"{events.height} events, {events.width} columns")
    display(events["type"].value_counts(sort=True))
    display(events.select("id", "type", "start", "end", "lat", "lon", "vessel_name", "vessel_flag").head(10))

## Was the result truncated?

The adapter sends a single request with `limit` and does **not** paginate, so a
result of exactly `limit` rows almost certainly means events were dropped.

In [ ]:
if events is not None and events.height >= LIMIT:
    print(f"⚠️  {events.height} events == limit ({LIMIT}): the response is probably truncated.")
    print("    Narrow the window/region, or raise `limit` on collekt.GFW(...).")
elif events is not None:
    print(f"{events.height} events < limit ({LIMIT}): the full result set.")

## Event positions

Each event carries a single representative position plus a bounding box.

In [ ]:
import matplotlib.pyplot as plt

if events is not None:
    located = events.filter(pl.col("lat").is_not_null() & pl.col("lon").is_not_null())
    fig, ax = plt.subplots(figsize=(9, 6))
    for event_type in located["type"].unique(maintain_order=True):
        subset = located.filter(pl.col("type") == event_type)
        ax.scatter(subset["lon"], subset["lat"], s=14, alpha=0.6, label=f"{event_type} ({subset.height})")
    west, east, south, north = request.region.west, request.region.east, request.region.south, request.region.north
    ax.plot([west, east, east, west, west], [south, south, north, north, south], "k--", lw=0.8, label="request bbox")
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")
    ax.set_title("GFW events, Balearic Sea, 1-7 June 2023")
    ax.legend(fontsize=8)
    plt.show()

## Type-specific detail

`encounter_json` / `fishing_json` / `gap_json` / `loitering_json` /
`port_visit_json` hold the per-type detail block as JSON text — exactly one is
populated per row, matching `type`. They are kept as text because GFW's response
models accept undocumented extra fields, which a fixed struct column would drop.

In [ ]:
import json

if events is not None:
    for column in ("fishing_json", "port_visit_json", "encounter_json", "loitering_json", "gap_json"):
        row = events.filter(pl.col(column).is_not_null()).head(1)
        if row.height:
            print(f"--- {column} ---")
            print(json.dumps(json.loads(row[column][0]), indent=2)[:600])
            print()

## Provenance

The Parquet file is a damast `AnnotatedDataFrame`: column descriptions and units
travel with the data.

In [ ]:
if result.files:
    import damast

    adf = damast.core.AnnotatedDataFrame.from_file(result.files[0])
    for annotation in adf.metadata.annotations.values():
        print(f"{annotation.name}: {annotation.value}")
    print()
    for spec in adf.metadata.columns:
        unit = f" [{spec.unit}]" if spec.unit else ""
        print(f"{spec.name}{unit}: {spec.description}")